# Pipeline de Ingestão: ADLS Gen2 ➔ Delta Lake (Raw)
Este notebook realiza:
- Autenticação via Service Principal com Azure Identity SDK.
- Varredura recursiva de diretórios para captura de novos arquivos `ecommerce_enderecos.parquet`.
- Download em memória e conversão para DataFrame PySpark.
- Tratamento básico de tipos e auditoria (`source_file`, `ingestion_time`).
- Carga incremental (`append`) com controle de idempotência/checkpoint via metadados.

In [0]:
import os
import io
import time
import pandas as pd
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import StringType

# ==============================================================================
# 1. Carregamento Seguro das Credenciais e Parâmetros
# ==============================================================================
load_dotenv(".env")

client_id     = os.getenv("ADLS_CLIENT_ID")
tenant_id     = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

storage_account_name = "internshipdatalake"
container_name = "raw"
base_directory = "real-time-data"
target_file_name = "ecommerce_enderecos.parquet"
delta_target_table = "raw_ecommerce_enderecos"

assert client_id and tenant_id and client_secret, "Credenciais do ADLS não encontradas no arquivo .env."

# ==============================================================================
# 2. Conexão com o Azure Data Lake Storage via SDK
# ==============================================================================
credential = ClientSecretCredential(
    tenant_id=tenant_id, 
    client_id=client_id, 
    client_secret=client_secret
)

service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account_name}.dfs.core.windows.net",
    credential=credential
)

file_system_client = service_client.get_file_system_client(file_system=container_name)
print("✔ Conexão com o ADLS via SDK estabelecida com sucesso.")

In [0]:
# ==============================================================================
# 3. Inicialização do Mecanismo de Checkpoint
# ==============================================================================
# Recupera histórico da tabela Delta para evitar reingestão de arquivos já processados
control_checkpoint = set()

try:
    if spark.catalog.tableExists(delta_target_table):
        processed_files = (
            spark.table(delta_target_table)
            .select("source_file")
            .distinct()
            .toPandas()["source_file"]
            .tolist()
        )
        control_checkpoint = set(processed_files)
        print(f"✔ Histórico carregado: {len(control_checkpoint)} arquivo(s) previamente ingerido(s).")
    else:
        print(f"ℹ Tabela '{delta_target_table}' não encontrada. Será inicializada na primeira ingestão.")
except Exception as e:
    print(f"⚠ Aviso ao verificar checkpoint da tabela: {e}")

In [0]:
# ==============================================================================
# 4. Loop de Monitoramento Contínuo (Polling & Ingestão)
# ==============================================================================
print(f"Iniciando monitoramento de '{target_file_name}' em '{base_directory}'...\n")

while True:
    try:
        paths = file_system_client.get_paths(path=base_directory, recursive=True)
        
        # Filtra apenas arquivos novos não processados
        new_files = [
            item.name for item in paths 
            if not item.is_directory and item.name.endswith(target_file_name) and item.name not in control_checkpoint
        ]
        
        if new_files:
            print(f"Detectado(s) {len(new_files)} novo(s) arquivo(s) para ingestão.")
            
            for file_path in new_files:
                print(f"Processando: {file_path}")
                
                # Download do arquivo direto para memória
                file_client = file_system_client.get_file_client(file_path)
                downloaded_bytes = file_client.download_file().readall()
                
                # Conversão Bytes -> Pandas -> PySpark DataFrame
                pdf = pd.read_parquet(io.BytesIO(downloaded_bytes))
                df_spark = spark.createDataFrame(pdf)

                # Padronização de tipagem: complemento para String
                if "complemento" in df_spark.columns:
                    df_spark = df_spark.withColumn("complemento", df_spark["complemento"].cast(StringType()))

                # Adição de colunas de auditoria e linhagem de dados
                df_spark = (
                    df_spark
                    .withColumn("source_file", lit(file_path))
                    .withColumn("ingestion_time", current_timestamp())
                )
                
                # Gravação incremental na tabela Delta Raw
                (
                    df_spark.write
                    .format("delta")
                    .mode("append")
                    .saveAsTable(delta_target_table)
                )
                
                control_checkpoint.add(file_path)
                print(f"✔ Sucesso: Arquivo '{file_path}' ingerido.")
                
        time.sleep(15)

    except KeyboardInterrupt:
        print("\nProcesso interrompido manualmente pelo usuário.")
        break
    except Exception as e:
        print(f"\n Erro durante a execução do loop: {e}")
        time.sleep(15)